# 03 - Geographic Target Encoding

EDA found `Order Country` carries a much wider late rate spread (about 30 percentage points) than `Order Region` (about 9 points), making country level delay rate the stronger feature candidate. We target encode it here.

**Leakage:** the encoding map (average delay rate per country) is fit **only on the training set**, then applied to validation and test unchanged. Fitting on the full dataset before splitting would leak future information into the encoding.


## Setup

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

TRAIN_IN = Path('../../../data/processed/features_step2_train.csv')
VAL_IN = Path('../../../data/processed/features_step2_val.csv')
TEST_IN = Path('../../../data/processed/features_step2_test.csv')

TRAIN_OUT = Path('../../../data/processed/features_step3_train.csv')
VAL_OUT = Path('../../../data/processed/features_step3_val.csv')
TEST_OUT = Path('../../../data/processed/features_step3_test.csv')

train_df = pd.read_csv(TRAIN_IN)
val_df = pd.read_csv(VAL_IN)
test_df = pd.read_csv(TEST_IN)


## 1. Fit the target encoding on the training set only

In [6]:
global_mean = train_df['Late_delivery_risk'].mean()

country_encoding_map = train_df.groupby('Order Country')['Late_delivery_risk'].mean()
region_encoding_map = train_df.groupby('Order Region')['Late_delivery_risk'].mean()

print(f"Global mean late rate (train): {global_mean:.4f}")
print(f"Countries seen in training data: {len(country_encoding_map)}")


Global mean late rate (train): 0.5485
Countries seen in training data: 164


## 2. Apply the encoding to all three splits

Any country/region appearing in validation or test that wasn't seen in training falls back to the global training mean - this is the standard, safe way to handle unseen categories in target encoding.


In [7]:
def apply_target_encoding(df, mapping, source_col, new_col, fallback):
    df = df.copy()
    df[new_col] = df[source_col].map(mapping).fillna(fallback)
    return df

for name, df_ref in [('train', train_df), ('val', val_df), ('test', test_df)]:
    n_unseen_country = (~df_ref['Order Country'].isin(country_encoding_map.index)).sum()
    print(f"{name}: {n_unseen_country} rows with an Order Country unseen in training")

train_df = apply_target_encoding(train_df, country_encoding_map, 'Order Country', 'country_delay_rate', global_mean)
val_df = apply_target_encoding(val_df, country_encoding_map, 'Order Country', 'country_delay_rate', global_mean)
test_df = apply_target_encoding(test_df, country_encoding_map, 'Order Country', 'country_delay_rate', global_mean)

train_df = apply_target_encoding(train_df, region_encoding_map, 'Order Region', 'region_delay_rate', global_mean)
val_df = apply_target_encoding(val_df, region_encoding_map, 'Order Region', 'region_delay_rate', global_mean)
test_df = apply_target_encoding(test_df, region_encoding_map, 'Order Region', 'region_delay_rate', global_mean)


train: 0 rows with an Order Country unseen in training
val: 0 rows with an Order Country unseen in training
test: 0 rows with an Order Country unseen in training


**What we found:**

**Full coverage, zero fallback needed.** All 164 countries present in the full dataset appear in the training split, and every row in validation and test maps to a country already seen during training, zero unseen category rows in either split. The fallback to global mean logic exists as a safety net and is correctly implemented, but in practice it never actually triggers here.

This is a good, clean result: it means `country_delay_rate` carries genuine, specific signal for every single order in validation and test, none of it diluted by falling back to the generic 0.5485 average. Worth noting in the report as a confirmation that the chronological split, despite being ordered by time rather than randomly shuffled, still preserved full geographic coverage across all three splits, since a real risk with a time-based split is that some rare category could disappear entirely from later time periods.


## 3. Save

In [8]:
train_df.to_csv(TRAIN_OUT, index=False)
val_df.to_csv(VAL_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)
print("Saved step 3 outputs.")


Saved step 3 outputs.


**`DECISION_LOG.md`:** target encoding fit strictly on training data, with a documented fallback strategy for unseen categories in validation/test.
